# Sessão 2: médias móveis, volatilidade e drawdown

**Meta de hoje:** calcular os quatro números que mais aparecem no app (retorno acumulado, média móvel, volatilidade e drawdown) e, no fim, **guardá-los em funções organizadas** em `src/indicadores.py`.

**Como usar este notebook** (igual à sessão 1)
- Rode cada célula em ordem com `Shift+Enter`.
- Células de **Exercício** têm `None  # TODO`: troque o `None` pelo seu código.
- A célula seguinte **confere** a resposta (✅ certo, ❌ errado, ⏳ ainda não respondido).
- Travou? Peça uma dica ao Claude: primeiro uma dica, depois uma dica maior, e só então a resposta comentada.

| Parte | O que vamos ver |
|---|---|
| 1 | Preparação e dados |
| 2 | Exercício D: retorno acumulado |
| 3 | Exercício E: médias móveis (`rolling`) |
| 4 | Exercício F: volatilidade |
| 5 | Exercício G: drawdown |
| 6 | Organização: criar `src/indicadores.py` |
| 7 | Fim da sessão |

## 1. Preparação e dados

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

print("Pandas versão:", pd.__version__)

# Constante com nome (prática de organização): nada de número solto no código
DIAS_POR_ANO = 252  # dias de negociação em um ano


def conferir(nome, resposta, esperado):
    """Compara sua resposta com a esperada e avisa o resultado."""
    if resposta is None:
        print(f"⏳ {nome}: ainda não respondido")
        return
    try:
        if isinstance(esperado, (float, np.floating)):
            certo = bool(np.isclose(float(resposta), float(esperado)))
        elif isinstance(esperado, list):
            certo = list(resposta) == esperado
        else:
            certo = bool(resposta == esperado)
    except (TypeError, ValueError):
        certo = False
    print(
        f"{'✅' if certo else '❌'} {nome}: {'certo!' if certo else 'confira de novo'}"
    )

Pandas versão: 3.0.6


Mesmos dados da sessão 1. Repare no `if ... raise` logo depois do download: é a validação que aprendemos com o aviso do Pylance.

In [2]:
ticker = "PETR4.SA"
dados = yf.download(ticker, period="2y", auto_adjust=True, progress=False)

if dados is None or dados.empty:
    raise ValueError(f"Não consegui baixar dados de {ticker}")

dados.columns = dados.columns.droplevel("Ticker")

fechamento = dados["Close"]
retorno = fechamento.pct_change()
fechamento.tail()


Date
2026-10-05    55.360001
2026-10-06    53.820000
2026-10-07    54.330002
2026-10-08    55.480000
2026-10-09    56.000000
Name: Close, dtype: float64

## 2. Exercício D: retorno acumulado

O **retorno diário** mostra um dia. O **retorno acumulado** responde: *"se eu tivesse comprado no começo, quanto teria ganhado até hoje?"*

Duas formas de chegar ao mesmo número:
1. **Pelos preços:** `preço final ÷ preço inicial − 1`.
2. **Pelos retornos:** multiplicar os fatores diários `(1 + retorno)`, dia após dia. Em Pandas: `(1 + retorno).cumprod()`, e depois `- 1`. O `cumprod()` é o produto *acumulado* (parecido com o `cumsum()`, que soma).

- **D1.** Retorno total do período, pela forma 1. *(dicas: `fechamento.iloc[-1]` e `fechamento.iloc[0]`)*
- **D2.** A série `acumulado` pela forma 2 (uma série, um valor por dia).
- **Pergunta para pensar:** o último valor da `acumulado` deve bater com o D1. Por quê?

In [ ]:
retorno_total = fechamento.iloc[-1] / fechamento.iloc[0] - 1
acumulado = (1 + retorno).cumprod() - 1

print(retorno_total)
acumulado.tail()

In [ ]:
# Conferência (não precisa mexer)
conferir("D1 retorno total", retorno_total, fechamento.iloc[-1] / fechamento.iloc[0] - 1)
conferir("D2 último acumulado", None if acumulado is None else acumulado.iloc[-1], fechamento.iloc[-1] / fechamento.iloc[0] - 1)

## 3. Exercício E: médias móveis

A **média móvel** suaviza o preço: em cada dia, é a média dos últimos *N* dias. A de 20 dias reage rápido; a de 50 dias é mais calma. Quando a curta cruza a longa, muita gente enxerga um sinal. **Isso será a nossa primeira estratégia de backtest.**

Em Pandas, a "janela deslizante" se chama `rolling`:

```python
fechamento.rolling(20).mean()
```

Leia como: *"para cada dia, olhe os 20 últimos e tire a média"*. Os primeiros 19 dias ficam `NaN`, porque ainda não há 20 dias de histórico.

- **E1.** `mm20`: média móvel de 20 dias do fechamento.
- **E2.** `mm50`: média móvel de 50 dias.
- **E3.** `dias_acima`: em quantos dias o fechamento ficou **acima** da `mm20`. *(dica: igual ao `retorno < 0` da sessão 1, só que comparando duas séries)*

In [5]:
mm20 = fechamento.rolling(window=20).mean()  # TODO
mm50 = fechamento.rolling(window=50).mean()  # TODO
dias_acima = (fechamento > mm20).sum()  # TODO

print(dias_acima)

295


In [6]:
# Conferência (não precisa mexer)
conferir("E1 última mm20", None if mm20 is None else mm20.iloc[-1], fechamento.iloc[-20:].mean())
conferir("E1 dias vazios da mm20", None if mm20 is None else int(mm20.isna().sum()), 19)
conferir("E2 última mm50", None if mm50 is None else mm50.iloc[-1], fechamento.iloc[-50:].mean())
conferir("E2 dias vazios da mm50", None if mm50 is None else int(mm50.isna().sum()), 49)
conferir("E3 dias acima da mm20", dias_acima, int((fechamento.iloc[19:] > fechamento.rolling(20).mean().iloc[19:]).sum()))

✅ E1 última mm20: certo!
✅ E1 dias vazios da mm20: certo!
✅ E2 última mm50: certo!
✅ E2 dias vazios da mm50: certo!
✅ E3 dias acima da mm20: certo!


**Para ver:** rode a célula abaixo e compare o gráfico com a sua intuição.

In [7]:
if mm20 is not None and mm50 is not None:
    fechamento.iloc[-250:].plot(label="Fechamento", figsize=(10, 4), legend=True)
    mm20.iloc[-250:].plot(label="MM 20", legend=True)
    mm50.iloc[-250:].plot(label="MM 50", legend=True)

ImportError: matplotlib is required for plotting when the default backend "matplotlib" is selected.

## 4. Exercício F: volatilidade

**Volatilidade** é o tamanho dos "sustos" do preço. Mede-se pelo **desvio padrão** dos retornos: quanto eles se afastam, em média, da própria média. Em Pandas, `.std()`.

O desvio padrão dos retornos *diários* é uma volatilidade **diária**. Para comparar com o que se vê no mercado, **anualiza-se**: multiplica-se por `√252` (a raiz do número de dias de negociação em um ano). Em Python: `np.sqrt(DIAS_POR_ANO)`.

- **F1.** `vol_diaria`: desvio padrão dos retornos diários.
- **F2.** `vol_anual`: a mesma volatilidade anualizada.
- **F3.** `vol_movel`: a volatilidade anualizada de uma **janela móvel de 20 dias** (série). *(dica: junte `rolling` da parte 3 com `std`)*

In [ ]:
vol_diaria = retorno.std()  # TODO
vol_anual = retorno.std() * np.sqrt(DIAS_POR_ANO)  # TODO
vol_movel = retorno.rolling(window=20).std() * np.sqrt(DIAS_POR_ANO)  # TODO

print(vol_diaria, vol_anual)

0.016120536954625068 0.25590559069658675


In [ ]:
# Conferência (não precisa mexer)
conferir("F1 vol diária", vol_diaria, retorno.dropna().std())
conferir("F2 vol anual", vol_anual, retorno.dropna().std() * np.sqrt(252))
conferir("F3 última vol móvel", None if vol_movel is None else vol_movel.iloc[-1], retorno.iloc[-20:].std() * np.sqrt(252))

✅ F1 vol diária: certo!
✅ F2 vol anual: certo!
✅ F3 última vol móvel: certo!


## 5. Exercício G: drawdown

**Drawdown** é *"quanto estou abaixo do meu melhor momento até agora"*. Se a ação já bateu R$ 100 e hoje vale R$ 80, o drawdown é −20%. O **máximo drawdown** é a pior queda do topo até o fundo: o número que mostra o susto que quem comprou na hora errada passou.

Receita em três passos:
1. **Pico até agora:** o maior preço visto até cada dia. É o `cummax()`, o máximo *acumulado* (primo do `cumprod()`).
2. **Drawdown de cada dia:** `fechamento ÷ pico − 1`. É sempre zero ou negativo.
3. **Máximo drawdown:** o menor valor dessa série, e a data em que aconteceu.

- **G1.** `pico`: série com o máximo acumulado.
- **G2.** `drawdown`: série com o drawdown de cada dia.
- **G3.** `maior_queda` e `data_fundo`: o valor e a data do pior drawdown.

In [ ]:
pico = fechamento.cummax()
drawdown = fechamento / pico - 1
maior_queda = drawdown.min()  # TODO
data_fundo = drawdown.idxmin()  # TODO

print(maior_queda, data_fundo)

-0.22161771519207907 2026-07-06 00:00:00


In [ ]:
# Conferência (não precisa mexer)
_pico = fechamento.expanding().max()
_dd = fechamento / _pico - 1
conferir("G1 último pico", None if pico is None else pico.iloc[-1], _pico.iloc[-1])
conferir("G2 último drawdown", None if drawdown is None else drawdown.iloc[-1], _dd.iloc[-1])
conferir("G3 maior queda", maior_queda, _dd.min())
conferir("G3 data do fundo", data_fundo, _dd.idxmin())

✅ G1 último pico: certo!
✅ G2 último drawdown: certo!
✅ G3 maior queda: certo!
✅ G3 data do fundo: certo!


## 6. Organização: do notebook para `src/indicadores.py`

O notebook é ótimo para **aprender**, mas o app precisa de código **reutilizável**. A prática de hoje: **separar o que calcula do que mostra.**

Vamos criar o arquivo `src/indicadores.py`, com **uma função para cada cálculo que você acabou de fazer**. Cada função:
- recebe uma Series (e, às vezes, um número) e devolve o resultado;
- tem uma **docstring** de uma linha;
- tem **nome que diz o que faz**, em português, com `_`;
- usa a **constante** `DIAS_POR_ANO`, em vez de um `252` solto.

Esqueleto (você escreve o corpo; use o código que fez acima):

```python
"""Indicadores de análise de ações: retorno, médias móveis, volatilidade e drawdown."""

import numpy as np
import pandas as pd

DIAS_POR_ANO = 252


def calcular_retorno_diario(fechamento: pd.Series) -> pd.Series:
    """Retorno percentual de cada dia em relação ao dia anterior."""
    ...  # TODO


def calcular_retorno_acumulado(retorno: pd.Series) -> pd.Series:
    """Retorno acumulado desde o primeiro dia."""
    ...  # TODO


def calcular_media_movel(fechamento: pd.Series, janela: int) -> pd.Series:
    """Média dos últimos `janela` dias."""
    ...  # TODO


def calcular_volatilidade_anual(retorno: pd.Series) -> float:
    """Desvio padrão dos retornos diários, anualizado."""
    ...  # TODO


def calcular_drawdown(fechamento: pd.Series) -> pd.Series:
    """Queda de cada dia em relação ao maior preço visto até ali."""
    ...  # TODO
```

**Quando terminar:** volte aqui e rode a próxima célula, que importa suas funções e confere se dão os mesmos números do notebook.

In [8]:
# Rode só depois de criar src/indicadores.py (não precisa mexer)
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from src import indicadores as ind

conferir("src: retorno diário", ind.calcular_retorno_diario(fechamento).iloc[-1], retorno.iloc[-1])
conferir("src: retorno acumulado", ind.calcular_retorno_acumulado(retorno).iloc[-1], fechamento.iloc[-1] / fechamento.iloc[0] - 1)
conferir("src: média móvel 20", ind.calcular_media_movel(fechamento, 20).iloc[-1], fechamento.iloc[-20:].mean())
conferir("src: volatilidade anual", ind.calcular_volatilidade_anual(retorno), retorno.dropna().std() * np.sqrt(252))
conferir("src: maior drawdown", ind.calcular_drawdown(fechamento).min(), (fechamento / fechamento.expanding().max() - 1).min())

✅ src: retorno diário: certo!
✅ src: retorno acumulado: certo!
✅ src: média móvel 20: certo!
✅ src: volatilidade anual: certo!
✅ src: maior drawdown: certo!


## 7. Fim da sessão

- [X] Rodou **Restart + Run All** sem erro?
- [x] `ruff check .` sem avisos?
- [x] `git status` mostra só o que você quer? Commits pequenos, ex.: `feat: calcula média móvel e volatilidade` e `refactor: move cálculos para src/indicadores.py`.
- [x] Diário: 3 linhas sobre o que aprendeu e o que ficou confuso.

**Pergunta para pensar:** por que o drawdown mostra um risco que a volatilidade sozinha esconde?